# **wiki2shards**

## Phase 0: Workbench (M0)

### Step 0.1: Create the skeleton and pin Python

```py
uv python pin 3.12.14
```
- The command `uv python pin 3.12.14` sets Python version 3.12.14 as the active Python version for your current working directory and project.
- Creates/Updates `.python-version`: It writes `3.12.14` to a `.python-version` file in the current directory.
- According to Charlie Marsh (creator of uv and founder of Astral), uv officially stands for "Ultraviolet".

```sh
mkdir -p src/wiki2shards/{io,acquire,extract,clean,tokenize,pack,loader,pipeline}
```
- `-p` stands for `parent` used in parent directory

```sh
touch src/wiki2shards/{io,acquire,extract,clean,tokenize,pack,loader,pipeline}/__init__.py
```

- creates a `__init__.py` inside every single directory within the curly brackets

```sh
echo "# wiki2shards" > README.md
```
- writes `# wiki2shards` into `README.md`

### Step 0.2: pyproject.toml

`pyproject.toml`
- `pyproject.toml` is the modern standard configuration file for Python projects. It centralizes project metadata, dependencies, and configuration for various development tools into a single file, replacing the older, fragmented ecosystem of setup.py, setup.cfg, and requirements.txt.

**`[project]`**
Declares the start of the core, standardized project metadata block (defined by PEP 621).

* **`name = "wiki2shards"`**
Sets the package's name. This is what you would use to pip install it.
* **`version = "0.1.0"`**
Sets the current version of the project.
* **`description = "Wikipedia dump -> checksummed token shards, plus a deterministic, resumable, mixture-aware loader"`**
Provides a short summary of the project's purpose.
* **`readme = "README.md"`**
Specifies that the long description (usually displayed on PyPI) should be pulled from the `README.md` file.
* **`requires-python = ">=3.12,<3.13"`**
Restricts this package to only run on Python 3.12.x. It will fail to install on Python 3.11 or Python 3.13.
* **`dependencies = [`**
Starts the list of core runtime dependencies required for the package to function.
* **`"numpy>=2.0",`**
Requires NumPy version 2.0 or higher.
* **`"mwparserfromhell==0.7.2",  # output bytes depend on it (wikitext goldens)`**
Pins this Wikipedia parsing library to exactly version 0.7.2. The comment explains this strict pinning ensures the processed output remains identical across builds.
* **`"tiktoken>=0.9",`**
Requires the `tiktoken` (OpenAI tokenizer) library version 0.9 or higher.
* **`"tokenizers==0.23.3",  # output bytes depend on it (fixture tokenizer)`**
Pins Hugging Face `tokenizers` to exactly 0.23.3 to guarantee deterministic tokenization.
* **`"pydantic>=2.7",`**
Requires Pydantic version 2.7 or higher for data validation.
* **`"typer>=0.12",`**
Requires Typer version 0.12 or higher to build the command-line interface.
* **`"pyyaml>=6.0",`**
Requires PyYAML version 6.0 or higher to parse YAML files.
* **`]`**
Closes the core dependencies list.

**`[project.scripts]`**
Defines console commands (CLI entry points) that get automatically installed with the package.

* **`wiki2shards = "wiki2shards.cli:main"`**
Creates a terminal command named `wiki2shards` that, when run, executes the `main` function inside the `wiki2shards.cli` Python module.

**`[dependency-groups]`**
Defines standard dependency groups (PEP 735) for different environments, natively supported by tools like `uv`.

* **`dev = [`**
Starts a group named `dev` containing dependencies only needed for developing the package, not running it.
* **`"pytest>=8",`**
Requires the Pytest testing framework version 8 or higher.
* **`"hypothesis>=6.100",`**
Requires Hypothesis v6.100+ for property-based testing.
* **`"pytest-timeout>=2.3",`**
Requires the pytest-timeout plugin v2.3+ to forcibly stop tests that hang.
* **`"ruff>=0.6",`**
Requires the Ruff linter/formatter version 0.6 or higher.
* **`"mypy>=1.11",`**
Requires the Mypy static type checker version 1.11 or higher.
* **`"types-PyYAML>=6.0",`**
Installs type hints for PyYAML so Mypy can properly type-check YAML-related code.
* **`]`**
Closes the dev dependencies list.

**`[build-system]`**
Defines the tools pip needs to build the package from source into a wheel (PEP 518).

* **`requires = ["hatchling"]`**
Specifies that the `hatchling` package must be downloaded before the build can start.
* **`build-backend = "hatchling.build"`**
Tells the build frontend (like pip) to use Hatchling's build system to package the code.

**`[tool.hatch.build.targets.wheel]`**
Configuration specific to the Hatchling build backend regarding how to build the `.whl` (wheel) file.

* **`packages = ["src/wiki2shards"]`**
Tells Hatchling that the actual Python code to be packaged lives in the `src/wiki2shards` directory (a standard `src` layout).

**`[tool.pytest.ini_options]`**
Configuration block for the Pytest testing framework.

* **`testpaths = ["tests"]`**
Tells Pytest to only look for tests inside the `tests` directory.
* **`pythonpath = ["tests"]`**
Adds the `tests` directory to the Python path during a test run, allowing test files to import helper functions from other test files.
* **`addopts = "-p no:cacheprovider --strict-markers"`**
Default command-line flags applied to every pytest run: disables test result caching (`-p no:cacheprovider`) and throws an error if an unregistered test marker is used (`--strict-markers`).
* **`timeout = 120`**
Instructs `pytest-timeout` to automatically fail any single test that takes longer than 120 seconds.
* **`markers = [`**
Starts a list to register custom test markers (decorators like `@pytest.mark.slow`).
* **`"slow: subprocess builds, crash matrix, SIGKILL resume (excluded from the fast suite)",`**
Registers the `slow` marker and provides a description of what it's used for.
* **`"bench: benchmarks, never gating",`**
Registers the `bench` marker used for benchmarking tests.
* **`]`**
Closes the markers list.

**`[tool.ruff]`**
Configuration block for the Ruff linter and formatter.

* **`line-length = 100`**
Sets the maximum allowed characters per line to 100 (overriding the default 88).
* **`target-version = "py312"`**
Tells Ruff to assume the code is running on Python 3.12, allowing it to apply modern formatting and linting rules specific to that version.

**`[tool.ruff.lint]`**
Specific configuration for Ruff's linting features.

* **`select = ["E", "F", "W", "I", "B", "UP", "SIM", "RUF"]`**
Enables specific families of linting rules: `E` and `W` (pycodestyle errors/warnings), `F` (Pyflakes), `I` (isort for imports), `B` (flake8-bugbear for common bugs), `UP` (pyupgrade for modern syntax), `SIM` (flake8-simplify), and `RUF` (Ruff's own rules).

**`[tool.mypy]`**
Configuration block for the Mypy static type checker.

* **`strict = true`**
Turns on Mypy's most restrictive mode, requiring type hints for everything and prohibiting dynamic typing loopholes.
* **`python_version = "3.12"`**
Tells Mypy to type-check against Python 3.12 rules.
* **`files = ["src"]`**
Instructs Mypy to only check the code inside the `src` directory (ignoring tests or scripts).

**`[[tool.mypy.overrides]]`**
Starts an override block (denoted by double brackets `[[ ]]` representing an array of tables in TOML) to apply specific Mypy rules to specific libraries.

* **`module = ["mwparserfromhell", "mwparserfromhell.*"]`**
Targets the `mwparserfromhell` library and all of its submodules for this override.
* **`ignore_missing_imports = true`**
Tells Mypy to suppress errors about this library missing type hints (since it does not provide them natively and no stub package exists).

### Step 0.3: Package marker, then install

`src/wiki2shards/__init__.py`

### Step 0.4: Crash-safe writes

`src/wiki2shards/io/atomic.py`

```python
"""Crash-safe writes: tmp -> fsync -> os.replace -> fsync(dir).

A reader can never observe a partially written file under its final name: the bytes land in a
hidden tmp file next to it, and only a complete, fsynced file is renamed into place. os.replace
is atomic on POSIX within one filesystem, which is why the tmp file lives in the same directory.
"""

```

* **Lines 1-6:** A module-level docstring summarizing the entire script's purpose. It explains the core mechanism of atomic file writes: data is written to a temporary file, synced to disk, and then atomically renamed over the original file so no other program ever sees a partially written state.

```python
from __future__ import annotations

```

* **Line 8:** Instructs Python to delay evaluating type hints. This allows you to use modern type hint syntax (like `Path | str`) even in older Python versions, and prevents circular import issues with typing.

```python
import os
from collections.abc import Iterator
from contextlib import contextmanager
from pathlib import Path
from typing import BinaryIO

```

* **Line 10:** Imports the standard `os` module, providing low-level operating system interfaces (like `fsync`, `replace`, `open`, `close`).
* **Line 11:** Imports `Iterator` for type hinting the return type of the context manager.
* **Line 12:** Imports the `contextmanager` decorator, which allows writing a function that behaves like a class with `__enter__` and `__exit__` methods for use in a `with` statement.
* **Line 13:** Imports `Path` from `pathlib` for modern, object-oriented filesystem path manipulation.
* **Line 14:** Imports `BinaryIO` for type hinting a file object opened in binary mode.

```python
TMP_SUFFIX = ".tmp"

```

* **Line 16:** Defines a constant string `.tmp` that will be appended to the end of temporary files.

```python
def tmp_name(final: Path) -> Path:
    """Hidden sibling used while `final` is being written."""
    return final.with_name(f".{final.name}{TMP_SUFFIX}")

```

* **Line 19:** Defines a function `tmp_name` that takes a `Path` object (the target file) and returns a new `Path` object.
* **Line 20:** A docstring explaining the function's purpose.
* **Line 21:** Generates the temporary file path. It takes the original filename (`final.name`), prepends a dot `.` to make it a hidden file on UNIX/Linux systems, and appends the `.tmp` suffix. It replaces the original filename in the original directory (`final.with_name(...)`).

```python
def fsync_dir(path: Path) -> None:
    """Persist a directory entry (the rename) itself."""
    fd = os.open(path, os.O_RDONLY)
    try:
        os.fsync(fd)
    finally:
        os.close(fd)

```

* **Line 24:** Defines a function `fsync_dir` that takes a directory path and returns nothing (`None`).
* **Line 25:** A docstring explaining that this persists the directory metadata (necessary to guarantee a file rename survives a power loss).
* **Line 26:** Uses the low-level `os.open` to open the directory itself in read-only mode (`os.O_RDONLY`), returning an integer file descriptor (`fd`).
* **Line 27:** Starts a `try` block to ensure cleanup happens even if an error occurs.
* **Line 28:** Calls `os.fsync()` on the directory's file descriptor, forcing the operating system to flush the directory's metadata changes to physical storage.
* **Line 29:** Starts the `finally` block, which executes regardless of whether the `try` block succeeded or threw an exception.
* **Line 30:** Closes the low-level file descriptor to prevent resource leaks.

```python
@contextmanager
def atomic_open(final: Path | str) -> Iterator[BinaryIO]:
    """Open a binary writer whose content appears at `final` only if the block succeeds."""

```

* **Line 33:** Applies the `@contextmanager` decorator, converting the generator function below into a context manager that can be used with a `with` statement.
* **Line 34:** Defines the `atomic_open` function, which accepts either a `Path` or a string, and yields a binary file object (`BinaryIO`).
* **Line 35:** A docstring explaining that the file will only appear at its final destination if the code block inside the `with` statement completes without errors.

```python
    final = Path(final)
    tmp = tmp_name(final)
    f = open(tmp, "wb")  # noqa: SIM115 - closed explicitly on both paths below

```

* **Line 36:** Normalizes the input `final` by ensuring it is a `Path` object (in case a string was passed).
* **Line 37:** Calls `tmp_name()` to determine the hidden temporary file path.
* **Line 38:** Opens the temporary file for writing in binary mode (`"wb"`). The `# noqa: SIM115` comment tells a specific linter (flake8-simplify) to ignore the warning about opening a file without using a `with` statement, since this function manually manages the closing.

```python
    try:
        yield f
        f.flush()
        os.fsync(f.fileno())
        f.close()
        os.replace(tmp, final)
        fsync_dir(final.parent)

```

* **Line 39:** Starts the `try` block for the "success" path.
* **Line 40:** `yield f` hands the open temporary file object back to the caller's `with` block. Execution pauses here until the user's `with` block finishes.
* **Line 41:** Once the user's block finishes successfully, execution resumes. `f.flush()` forces Python's internal memory buffers to write to the OS.
* **Line 42:** `os.fsync(f.fileno())` tells the OS to physically write its own disk buffers to the hardware storage, guaranteeing the data is saved.
* **Line 43:** Closes the temporary file. It is now safely written to disk, but still has the `.tmp` name.
* **Line 44:** `os.replace` atomically renames the temporary file to the final filename. If a file already exists at `final`, it is silently and atomically overwritten.
* **Line 45:** Calls the previously defined `fsync_dir` on the parent directory (`final.parent`) to guarantee that the OS persists the file rename to disk.

```python
    except BaseException:
        f.close()
        tmp.unlink(missing_ok=True)
        raise

```

* **Line 46:** The `except BaseException` block catches *any* exception that occurred while the user was interacting with the yielded file in the `with` block, including system-exiting exceptions like `KeyboardInterrupt`.
* **Line 47:** Closes the file object to release the resource.
* **Line 48:** Deletes the temporary file (`unlink`). `missing_ok=True` prevents a `FileNotFoundError` if the file somehow doesn't exist.
* **Line 49:** `raise` re-throws the original exception so the calling program knows an error occurred and can handle it or crash normally.

```python
def atomic_write_bytes(final: Path | str, data: bytes) -> None:
    with atomic_open(final) as f:
        f.write(data)

```

* **Line 52:** Defines a helper function that takes a target path and a payload of raw bytes, returning nothing.
* **Line 53:** Uses the `atomic_open` context manager defined above to safely open the destination file.
* **Line 54:** Writes the entire `data` byte payload into the yielded file object. Once this block ends, `atomic_open` handles flushing, syncing, and renaming.

```python
def commit(src: Path, final: Path) -> None:
    """Move an already-complete file (written elsewhere in the same filesystem) into place."""
    with open(src, "rb") as f:
        os.fsync(f.fileno())
    os.replace(src, final)
    fsync_dir(final.parent)

```

* **Line 57:** Defines a `commit` function intended to take an existing file (`src`) and safely finalize it as `final`.
* **Line 58:** A docstring explaining its use case.
* **Line 59:** Opens the source file in read-binary mode using a standard `with` context manager.
* **Line 60:** Calls `os.fsync()` on the source file to ensure any pending writes made by whoever created the file are physically on disk before moving it.
* **Line 61:** The `with` block ends, automatically closing the source file.
* **Line 62:** Atomically renames the source file to its final destination.
* **Line 63:** Syncs the destination's parent directory to ensure the rename operation is permanently recorded on disk.